# PatchCore Review Notebook (WideResNet50-2 layer3, x224)

This notebook is the curated reproducibility and review notebook for the saved `x224` WRN50-2 layer3 PatchCore sweep.

Default behavior:
- load the checked-in sweep CSVs and per-variant score files
- recompute confusion matrices and defect analysis from the local `x224` metadata
- save regenerated figures into each variant folder without retraining

Optional execution mode:
-

## Imports and Paths

This cell loads the shared evaluation helpers, resolves the local artifact locations, and exposes the rerun flags.

In [ ]:
try:
    from pathlib import Path
    import json
    import subprocess
    import sys
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from IPython.display import display
    cwd = Path.cwd().resolve()
    candidate_roots = [cwd, *cwd.parents]
    REPO_ROOT = None
    for candidate in candidate_roots:
        if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():
            REPO_ROOT = candidate
            break
    if REPO_ROOT is None:
        raise RuntimeError('Could not locate repo root containing src/wafer_defect and configs/')
    SRC_ROOT = REPO_ROOT / 'src'
    if str(SRC_ROOT) not in sys.path:
        sys.path.insert(0, str(SRC_ROOT))
    from wafer_defect.evaluation import summarize_threshold_metrics
    ARTIFACT_ROOT = REPO_ROOT / 'experiments/anomaly_detection/patchcore/wideresnet50/x224/layer3/artifacts/patchcore-wideresnet50-layer3'
    RESULTS_DIR = ARTIFACT_ROOT / 'results'
    PLOTS_DIR = ARTIFACT_ROOT / 'plots'
    METADATA_PATH = REPO_ROOT / 'data/processed/x224/wm811k/metadata_50k_5pct.csv'
    RUNNER_PATH = REPO_ROOT / 'scripts/run_patchcore_wrn50_x224_umap.py'
    DATA_CONFIG_PATH = REPO_ROOT / 'data/dataset/x224/benchmark_50k_5pct/data_config.toml'
    TRAIN_CONFIG_PATH = REPO_ROOT / 'experiments/anomaly_detection/patchcore/wideresnet50/x224/layer3/train_config.toml'
    RAW_PICKLE_PATH = REPO_ROOT / 'data/raw/LSWMD.pkl'
    RETRAIN = False
    RETRAIN_SKIP_UMAP = False
    NUM_WORKERS = 4
    SELECTED_VARIANT_NAME = None
    RENDER_ALL_CACHED_VARIANTS = True
    VARIANTS_TO_RENDER: list[str] = []
    VARIANT_COLOR_VAL = '#4d908e'
    VARIANT_COLOR_NORMAL = '#577590'
    VARIANT_COLOR_ANOMALY = '#e07a5f'
    VARIANT_COLOR_DEFECT = '#81b29a'
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "from pathlib import path\nimport json\nimport subprocess\nimport sys\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom ipython.display import display\ncwd = path.cwd().resolve()\ncandidate_roots = [cwd, *cwd.parents]\nrepo_root = none\nfor candidate in candidate_roots:\n    if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():\n        repo_root = candidate\n        break\nif repo_root is none:\n    raise runtimeerror('could not locate repo root containing src/wafer_defect and configs/')\nsrc_root = repo_root / 'src'\nif str(src_root) not in sys.path:\n    sys.path.insert(0, str(src_root))\nfrom wafer_defect.evaluation import summarize_threshold_metrics\nartifact_root = repo_root / 'experiments/anomaly_detection/patchcore/wideresnet50/x224/layer3/artifacts/patchcore-wideresnet50-layer3'\nresults_dir = artifact_root / 'results'\nplots_dir = artifact_root / 'plots'\nmetadata_path = repo_root / 'data/processed/x224/wm811k/metadata_50k_5pct.csv'\nrunner_path = repo_root / 'scripts/run_patchcore_wrn50_x224_umap.py'\ndata_config_path = repo_root / 'data/dataset/x224/benchmark_50k_5pct/data_config.toml'\ntrain_config_path = repo_root / 'experiments/anomaly_detection/patchcore/wideresnet50/x224/layer3/train_config.toml'\nraw_pickle_path = repo_root / 'data/raw/lswmd.pkl'\nretrain = false\nretrain_skip_umap = false\nnum_workers = 4\nselected_variant_name = none\nrender_all_cached_variants = true\nvariants_to_render: list[str] = []\nvariant_color_val = '#4d908e'\nvariant_color_normal = '#577590'\nvariant_color_anomaly = '#e07a5f'\nvariant_color_defect = '#81b29a'"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Optional Full Rerun

In [ ]:
try:
    if RETRAIN:
        command = [sys.executable, '-u', str(RUNNER_PATH), '--raw-pickle', str(RAW_PICKLE_PATH), '--data-config', str(DATA_CONFIG_PATH), '--train-config', str(TRAIN_CONFIG_PATH), '--output-dir', str(ARTIFACT_ROOT), '--num-workers', str(NUM_WORKERS)]
        if RETRAIN_SKIP_UMAP:
            command.append('--skip-umap')
        print('Launching full rerun with the same runner used by the Modal app:')
        print(' '.join(command))
        subprocess.run(command, check=True, cwd=REPO_ROOT)
    else:
        print('RETRAIN = False, using saved artifacts from the canonical artifact root.')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "if retrain:\n    command = [sys.executable, '-u', str(runner_path), '--raw-pickle', str(raw_pickle_path), '--data-config', str(data_config_path), '--train-config', str(train_config_path), '--output-dir', str(artifact_root), '--num-workers', str(num_workers)]\n    if retrain_skip_umap:\n        command.append('--skip-umap')\n    print('launching full rerun with the same runner used by the modal app:')\n    print(' '.join(command))\n    subprocess.run(command, check=true, cwd=repo_root)\nelse:\n    print('retrain = false, using saved artifacts from the canonical artifact root.')"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Metadata and Sweep Loading

This cell loads the saved sweep table, selects the best variant by default, and attaches the local `x224` benchmark metadata for downstream analysis.

In [ ]:
try:
    metadata = pd.read_csv(METADATA_PATH)
    test_metadata = metadata[metadata['split'] == 'test'].reset_index(drop=True)
    sweep_results_df = pd.read_csv(RESULTS_DIR / 'patchcore_sweep_results.csv')
    sweep_summary = json.loads((RESULTS_DIR / 'patchcore_sweep_summary.json').read_text(encoding='utf-8'))
    selected_variant_name = str(SELECTED_VARIANT_NAME or sweep_summary['best_variant']['name'])
    display(metadata['split'].value_counts().rename_axis('split').to_frame('count'))
    display(sweep_results_df)
    print(f'Selected variant: {selected_variant_name}')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "metadata = pd.read_csv(metadata_path)\ntest_metadata = metadata[metadata['split'] == 'test'].reset_index(drop=true)\nsweep_results_df = pd.read_csv(results_dir / 'patchcore_sweep_results.csv')\nsweep_summary = json.loads((results_dir / 'patchcore_sweep_summary.json').read_text(encoding='utf-8'))\nselected_variant_name = str(selected_variant_name or sweep_summary['best_variant']['name'])\ndisplay(metadata['split'].value_counts().rename_axis('split').to_frame('count'))\ndisplay(sweep_results_df)\nprint(f'selected variant: {selected_variant_name}')"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Variant Loaders

These helpers normalize the per-variant artifact layout, recompute threshold metrics from the saved score CSVs, and render plots back into the variant folders.

In [ ]:
try:

    def load_variant_outputs(variant_name: str) -> dict[str, object]:
        variant_root = ARTIFACT_ROOT / variant_name
        summary_path = variant_root / 'results' / 'summary.json'
        if not summary_path.exists():
            raise FileNotFoundError(f'Missing summary for {variant_name}: {summary_path}')
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        val_scores_df = pd.read_csv(variant_root / 'results' / 'evaluation' / 'val_scores.csv')
        test_scores_df = pd.read_csv(variant_root / 'results' / 'evaluation' / 'test_scores.csv')
        threshold_sweep_df = pd.read_csv(variant_root / 'results' / 'evaluation' / 'threshold_sweep.csv')
        threshold = float(summary['threshold'])
        metrics = summarize_threshold_metrics(test_scores_df['is_anomaly'].to_numpy(), test_scores_df['score'].to_numpy(), threshold)
        best_sweep = threshold_sweep_df.sort_values('f1', ascending=False).iloc[0].to_dict()
        defect_breakdown_path = variant_root / 'results' / 'evaluation' / 'selected_defect_breakdown.csv'
        defect_breakdown_df = pd.read_csv(defect_breakdown_path) if defect_breakdown_path.exists() else None
        return {'summary': summary, 'val_scores_df': val_scores_df, 'test_scores_df': test_scores_df, 'threshold_sweep_df': threshold_sweep_df, 'metrics': metrics, 'best_sweep': best_sweep, 'variant_root': variant_root, 'defect_breakdown_df': defect_breakdown_df}

    def compute_failure_tables(test_metadata: pd.DataFrame, test_scores_df: pd.DataFrame, threshold: float) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        analysis_df = test_metadata.copy()
        analysis_df['score'] = test_scores_df.reset_index(drop=True)['score']
        analysis_df['predicted_anomaly'] = (analysis_df['score'] > threshold).astype(int)
        analysis_df['error_type'] = 'tn'
        analysis_df.loc[(analysis_df['is_anomaly'] == 0) & (analysis_df['predicted_anomaly'] == 1), 'error_type'] = 'fp'
        analysis_df.loc[(analysis_df['is_anomaly'] == 1) & (analysis_df['predicted_anomaly'] == 0), 'error_type'] = 'fn'
        analysis_df.loc[(analysis_df['is_anomaly'] == 1) & (analysis_df['predicted_anomaly'] == 1), 'error_type'] = 'tp'
        error_summary_df = analysis_df.groupby('error_type').agg(count=('error_type', 'size'), mean_score=('score', 'mean')).reindex(['tp', 'fn', 'fp', 'tn'])
        defect_recall_df = analysis_df[analysis_df['is_anomaly'] == 1].groupby('defect_type').agg(count=('defect_type', 'size'), detected=('predicted_anomaly', 'sum'), mean_score=('score', 'mean')).sort_values(['detected', 'count'], ascending=[False, False])
        defect_recall_df['recall'] = defect_recall_df['detected'] / defect_recall_df['count']
        return (analysis_df, error_summary_df, defect_recall_df)

    def render_variant_artifacts(variant_name: str, payload: dict[str, object]) -> dict[str, str]:
        summary = payload['summary']
        threshold = float(summary['threshold'])
        val_scores_df = payload['val_scores_df']
        test_scores_df = payload['test_scores_df']
        threshold_sweep_df = payload['threshold_sweep_df']
        metrics = payload['metrics']
        best_sweep = payload['best_sweep']
        variant_root = payload['variant_root']
        variant_plots_dir = variant_root / 'plots'
        variant_eval_dir = variant_root / 'results' / 'evaluation'
        variant_plots_dir.mkdir(exist_ok=True)
        variant_eval_dir.mkdir(parents=True, exist_ok=True)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].hist(val_scores_df['score'], bins=30, alpha=0.85, color=VARIANT_COLOR_VAL)
        axes[0].axvline(threshold, color='red', linestyle='--', label=f'threshold={threshold:.4f}')
        axes[0].set_title(f'Validation Normal Score Distribution\n{variant_name}')
        axes[0].legend()
        axes[1].hist(test_scores_df[test_scores_df['is_anomaly'] == 0]['score'], bins=30, alpha=0.7, label='normal', color=VARIANT_COLOR_NORMAL)
        axes[1].hist(test_scores_df[test_scores_df['is_anomaly'] == 1]['score'], bins=30, alpha=0.7, label='anomaly', color=VARIANT_COLOR_ANOMALY)
        axes[1].axvline(threshold, color='red', linestyle='--', label=f'threshold={threshold:.4f}')
        axes[1].set_title(f'Test Score Distribution\n{variant_name}')
        axes[1].legend()
        plt.tight_layout()
        fig.savefig(variant_plots_dir / 'score_distribution.png', dpi=200, bbox_inches='tight')
        plt.close(fig)
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['precision'], label='precision')
        ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['recall'], label='recall')
        ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['f1'], label='f1')
        ax.axvline(threshold, color='red', linestyle='--', label=f'validation threshold = {threshold:.4f}')
        ax.axvline(best_sweep['threshold'], color='green', linestyle=':', label=f"best sweep threshold = {best_sweep['threshold']:.4f}")
        ax.set_title(f'Threshold Sweep on Test Split\n{variant_name}')
        ax.set_xlabel('Anomaly-score threshold')
        ax.set_ylabel('Metric value')
        ax.legend()
        plt.tight_layout()
        fig.savefig(variant_plots_dir / 'threshold_sweep.png', dpi=200, bbox_inches='tight')
        plt.close(fig)
        cm_array = np.asarray(metrics['confusion_matrix'], dtype=float)
        fig, ax = plt.subplots(figsize=(5, 4))
        im = ax.imshow(cm_array, cmap='Blues')
        ax.set_xticks([0, 1], labels=['pred_normal', 'pred_anomaly'])
        ax.set_yticks([0, 1], labels=['true_normal', 'true_anomaly'])
        ax.set_title(f'Confusion Matrix\n{variant_name}')
        for row_idx in range(cm_array.shape[0]):
            for col_idx in range(cm_array.shape[1]):
                ax.text(col_idx, row_idx, int(cm_array[row_idx, col_idx]), ha='center', va='center', color='black')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        plt.tight_layout()
        fig.savefig(variant_plots_dir / 'confusion_matrix.png', dpi=200, bbox_inches='tight')
        plt.close(fig)
        analysis_df, error_summary_df, defect_recall_df = compute_failure_tables(test_metadata, test_scores_df, threshold)
        analysis_df.to_csv(variant_eval_dir / 'analysis_with_predictions.csv', index=False)
        error_summary_df.reset_index().to_csv(variant_eval_dir / 'error_summary.csv', index=False)
        defect_recall_df.reset_index().to_csv(variant_eval_dir / 'defect_recall.csv', index=False)
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        axes[0].bar(error_summary_df.index.astype(str), error_summary_df['count'], color=VARIANT_COLOR_ANOMALY)
        axes[0].set_title(f'Prediction Outcome Counts\n{variant_name}')
        axes[0].set_ylabel('count')
        top_defects_df = defect_recall_df.head(10).reset_index()
        axes[1].barh(top_defects_df['defect_type'], top_defects_df['recall'], color=VARIANT_COLOR_DEFECT)
        axes[1].set_xlim(0.0, 1.0)
        axes[1].set_title('Top Defect-Type Recall')
        axes[1].set_xlabel('recall')
        axes[1].invert_yaxis()
        plt.tight_layout()
        fig.savefig(variant_plots_dir / 'defect_breakdown.png', dpi=200, bbox_inches='tight')
        plt.close(fig)
        return {'plots_dir': str(variant_plots_dir), 'evaluation_dir': str(variant_eval_dir)}
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "def load_variant_outputs(variant_name: str) -> dict[str, object]:\n    variant_root = artifact_root / variant_name\n    summary_path = variant_root / 'results' / 'summary.json'\n    if not summary_path.exists():\n        raise filenotfounderror(f'missing summary for {variant_name}: {summary_path}')\n    summary = json.loads(summary_path.read_text(encoding='utf-8'))\n    val_scores_df = pd.read_csv(variant_root / 'results' / 'evaluation' / 'val_scores.csv')\n    test_scores_df = pd.read_csv(variant_root / 'results' / 'evaluation' / 'test_scores.csv')\n    threshold_sweep_df = pd.read_csv(variant_root / 'results' / 'evaluation' / 'threshold_sweep.csv')\n    threshold = float(summary['threshold'])\n    metrics = summarize_threshold_metrics(test_scores_df['is_anomaly'].to_numpy(), test_scores_df['score'].to_numpy(), threshold)\n    best_sweep = threshold_sweep_df.sort_values('f1', ascending=false).iloc[0].to_dict()\n    defect_breakdown_path = variant_root / 'results' / 'evaluation' / 'selected_defect_breakdown.csv'\n    defect_breakdown_df = pd.read_csv(defect_breakdown_path) if defect_breakdown_path.exists() else none\n    return {'summary': summary, 'val_scores_df': val_scores_df, 'test_scores_df': test_scores_df, 'threshold_sweep_df': threshold_sweep_df, 'metrics': metrics, 'best_sweep': best_sweep, 'variant_root': variant_root, 'defect_breakdown_df': defect_breakdown_df}\n\ndef compute_failure_tables(test_metadata: pd.dataframe, test_scores_df: pd.dataframe, threshold: float) -> tuple[pd.dataframe, pd.dataframe, pd.dataframe]:\n    analysis_df = test_metadata.copy()\n    analysis_df['score'] = test_scores_df.reset_index(drop=true)['score']\n    analysis_df['predicted_anomaly'] = (analysis_df['score'] > threshold).astype(int)\n    analysis_df['error_type'] = 'tn'\n    analysis_df.loc[(analysis_df['is_anomaly'] == 0) & (analysis_df['predicted_anomaly'] == 1), 'error_type'] = 'fp'\n    analysis_df.loc[(analysis_df['is_anomaly'] == 1) & (analysis_df['predicted_anomaly'] == 0), 'error_type'] = 'fn'\n    analysis_df.loc[(analysis_df['is_anomaly'] == 1) & (analysis_df['predicted_anomaly'] == 1), 'error_type'] = 'tp'\n    error_summary_df = analysis_df.groupby('error_type').agg(count=('error_type', 'size'), mean_score=('score', 'mean')).reindex(['tp', 'fn', 'fp', 'tn'])\n    defect_recall_df = analysis_df[analysis_df['is_anomaly'] == 1].groupby('defect_type').agg(count=('defect_type', 'size'), detected=('predicted_anomaly', 'sum'), mean_score=('score', 'mean')).sort_values(['detected', 'count'], ascending=[false, false])\n    defect_recall_df['recall'] = defect_recall_df['detected'] / defect_recall_df['count']\n    return (analysis_df, error_summary_df, defect_recall_df)\n\ndef render_variant_artifacts(variant_name: str, payload: dict[str, object]) -> dict[str, str]:\n    summary = payload['summary']\n    threshold = float(summary['threshold'])\n    val_scores_df = payload['val_scores_df']\n    test_scores_df = payload['test_scores_df']\n    threshold_sweep_df = payload['threshold_sweep_df']\n    metrics = payload['metrics']\n    best_sweep = payload['best_sweep']\n    variant_root = payload['variant_root']\n    variant_plots_dir = variant_root / 'plots'\n    variant_eval_dir = variant_root / 'results' / 'evaluation'\n    variant_plots_dir.mkdir(exist_ok=true)\n    variant_eval_dir.mkdir(parents=true, exist_ok=true)\n    fig, axes = plt.subplots(1, 2, figsize=(12, 4))\n    axes[0].hist(val_scores_df['score'], bins=30, alpha=0.85, color=variant_color_val)\n    axes[0].axvline(threshold, color='red', linestyle='--', label=f'threshold={threshold:.4f}')\n    axes[0].set_title(f'validation normal score distribution\\n{variant_name}')\n    axes[0].legend()\n    axes[1].hist(test_scores_df[test_scores_df['is_anomaly'] == 0]['score'], bins=30, alpha=0.7, label='normal', color=variant_color_normal)\n    axes[1].hist(test_scores_df[test_scores_df['is_anomaly'] == 1]['score'], bins=30, alpha=0.7, label='anomaly', color=variant_color_anomaly)\n    axes[1].axvline(threshold, color='red', linesty"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Selected Variant Review

This cell loads the selected variant, shows the key metrics, and saves the main review plots into the branch-level `plots/` folder.

In [ ]:
try:
    selected_variant = load_variant_outputs(selected_variant_name)
    summary = selected_variant['summary']
    val_scores_df = selected_variant['val_scores_df']
    test_scores_df = selected_variant['test_scores_df']
    threshold_sweep_df = selected_variant['threshold_sweep_df']
    metrics = selected_variant['metrics']
    best_sweep = selected_variant['best_sweep']
    threshold = float(summary['threshold'])
    metrics_df = pd.DataFrame([{'metric': 'precision', 'value': metrics['precision']}, {'metric': 'recall', 'value': metrics['recall']}, {'metric': 'f1', 'value': metrics['f1']}, {'metric': 'auroc', 'value': metrics['auroc']}, {'metric': 'auprc', 'value': metrics['auprc']}, {'metric': 'threshold', 'value': threshold}])
    confusion_df = pd.DataFrame(metrics['confusion_matrix'], index=['true_normal', 'true_anomaly'], columns=['pred_normal', 'pred_anomaly'])
    display(metrics_df)
    display(confusion_df)
    plot_df = sweep_results_df.copy().sort_values(['f1', 'auroc'], ascending=False).reset_index(drop=True)
    plot_df['label'] = plot_df['name']
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].barh(plot_df['label'], plot_df['f1'], color='#3d405b')
    axes[0].set_title('WRN50-2 layer3 PatchCore: F1')
    axes[0].invert_yaxis()
    axes[1].barh(plot_df['label'], plot_df['auroc'], color='#81b29a')
    axes[1].set_title('WRN50-2 layer3 PatchCore: AUROC')
    axes[1].invert_yaxis()
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'variant_comparison_metrics.png', dpi=200, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(val_scores_df['score'], bins=30, alpha=0.85, color=VARIANT_COLOR_VAL)
    axes[0].axvline(threshold, color='red', linestyle='--', label=f'threshold={threshold:.4f}')
    axes[0].set_title(f'Validation Normal Score Distribution\n{selected_variant_name}')
    axes[0].legend()
    axes[1].hist(test_scores_df[test_scores_df['is_anomaly'] == 0]['score'], bins=30, alpha=0.7, label='normal', color=VARIANT_COLOR_NORMAL)
    axes[1].hist(test_scores_df[test_scores_df['is_anomaly'] == 1]['score'], bins=30, alpha=0.7, label='anomaly', color=VARIANT_COLOR_ANOMALY)
    axes[1].axvline(threshold, color='red', linestyle='--', label=f'threshold={threshold:.4f}')
    axes[1].set_title(f'Test Score Distribution\n{selected_variant_name}')
    axes[1].legend()
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'score_distribution.png', dpi=200, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['precision'], label='precision')
    ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['recall'], label='recall')
    ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['f1'], label='f1')
    ax.axvline(threshold, color='red', linestyle='--', label=f'validation threshold = {threshold:.4f}')
    ax.axvline(best_sweep['threshold'], color='green', linestyle=':', label=f"best sweep threshold = {best_sweep['threshold']:.4f}")
    ax.set_title(f'Threshold Sweep on Test Split\n{selected_variant_name}')
    ax.legend()
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'threshold_sweep.png', dpi=200, bbox_inches='tight')
    plt.show()
    plt.close(fig)
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = 'selected_variant = load_variant_outputs(selected_variant_name)\nsummary = selected_variant[\'summary\']\nval_scores_df = selected_variant[\'val_scores_df\']\ntest_scores_df = selected_variant[\'test_scores_df\']\nthreshold_sweep_df = selected_variant[\'threshold_sweep_df\']\nmetrics = selected_variant[\'metrics\']\nbest_sweep = selected_variant[\'best_sweep\']\nthreshold = float(summary[\'threshold\'])\nmetrics_df = pd.dataframe([{\'metric\': \'precision\', \'value\': metrics[\'precision\']}, {\'metric\': \'recall\', \'value\': metrics[\'recall\']}, {\'metric\': \'f1\', \'value\': metrics[\'f1\']}, {\'metric\': \'auroc\', \'value\': metrics[\'auroc\']}, {\'metric\': \'auprc\', \'value\': metrics[\'auprc\']}, {\'metric\': \'threshold\', \'value\': threshold}])\nconfusion_df = pd.dataframe(metrics[\'confusion_matrix\'], index=[\'true_normal\', \'true_anomaly\'], columns=[\'pred_normal\', \'pred_anomaly\'])\ndisplay(metrics_df)\ndisplay(confusion_df)\nplot_df = sweep_results_df.copy().sort_values([\'f1\', \'auroc\'], ascending=false).reset_index(drop=true)\nplot_df[\'label\'] = plot_df[\'name\']\nfig, axes = plt.subplots(1, 2, figsize=(14, 5))\naxes[0].barh(plot_df[\'label\'], plot_df[\'f1\'], color=\'#3d405b\')\naxes[0].set_title(\'wrn50-2 layer3 patchcore: f1\')\naxes[0].invert_yaxis()\naxes[1].barh(plot_df[\'label\'], plot_df[\'auroc\'], color=\'#81b29a\')\naxes[1].set_title(\'wrn50-2 layer3 patchcore: auroc\')\naxes[1].invert_yaxis()\nplt.tight_layout()\nfig.savefig(plots_dir / \'variant_comparison_metrics.png\', dpi=200, bbox_inches=\'tight\')\nplt.show()\nplt.close(fig)\nfig, axes = plt.subplots(1, 2, figsize=(12, 4))\naxes[0].hist(val_scores_df[\'score\'], bins=30, alpha=0.85, color=variant_color_val)\naxes[0].axvline(threshold, color=\'red\', linestyle=\'--\', label=f\'threshold={threshold:.4f}\')\naxes[0].set_title(f\'validation normal score distribution\\n{selected_variant_name}\')\naxes[0].legend()\naxes[1].hist(test_scores_df[test_scores_df[\'is_anomaly\'] == 0][\'score\'], bins=30, alpha=0.7, label=\'normal\', color=variant_color_normal)\naxes[1].hist(test_scores_df[test_scores_df[\'is_anomaly\'] == 1][\'score\'], bins=30, alpha=0.7, label=\'anomaly\', color=variant_color_anomaly)\naxes[1].axvline(threshold, color=\'red\', linestyle=\'--\', label=f\'threshold={threshold:.4f}\')\naxes[1].set_title(f\'test score distribution\\n{selected_variant_name}\')\naxes[1].legend()\nplt.tight_layout()\nfig.savefig(plots_dir / \'score_distribution.png\', dpi=200, bbox_inches=\'tight\')\nplt.show()\nplt.close(fig)\nfig, ax = plt.subplots(figsize=(8, 4))\nax.plot(threshold_sweep_df[\'threshold\'], threshold_sweep_df[\'precision\'], label=\'precision\')\nax.plot(threshold_sweep_df[\'threshold\'], threshold_sweep_df[\'recall\'], label=\'recall\')\nax.plot(threshold_sweep_df[\'threshold\'], threshold_sweep_df[\'f1\'], label=\'f1\')\nax.axvline(threshold, color=\'red\', linestyle=\'--\', label=f\'validation threshold = {threshold:.4f}\')\nax.axvline(best_sweep[\'threshold\'], color=\'green\', linestyle=\':\', label=f"best sweep threshold = {best_sweep[\'threshold\']:.4f}")\nax.set_title(f\'threshold sweep on test split\\n{selected_variant_name}\')\nax.legend()\nplt.tight_layout()\nfig.savefig(plots_dir / \'threshold_sweep.png\', dpi=200, bbox_inches=\'tight\')\nplt.show()\nplt.close(fig)'
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Failure Analysis and Cached Variant Rendering

This section computes the selected variant's defect-level behavior and then regenerates the same outputs for every saved variant folder from cached CSVs.

In [ ]:
try:
    analysis_df, error_summary_df, defect_recall_df = compute_failure_tables(test_metadata, test_scores_df, threshold)
    analysis_df.to_csv(RESULTS_DIR / 'selected_analysis_with_predictions.csv', index=False)
    error_summary_df.reset_index().to_csv(RESULTS_DIR / 'selected_error_summary.csv', index=False)
    defect_recall_df.reset_index().to_csv(RESULTS_DIR / 'selected_defect_recall.csv', index=False)
    display(error_summary_df)
    display(defect_recall_df)
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    axes[0].bar(error_summary_df.index.astype(str), error_summary_df['count'], color=VARIANT_COLOR_ANOMALY)
    axes[0].set_title(f'Prediction Outcome Counts\n{selected_variant_name}')
    axes[0].set_ylabel('count')
    top_defects_df = defect_recall_df.head(10).reset_index()
    axes[1].barh(top_defects_df['defect_type'], top_defects_df['recall'], color=VARIANT_COLOR_DEFECT)
    axes[1].set_xlim(0.0, 1.0)
    axes[1].set_title('Top Defect-Type Recall')
    axes[1].set_xlabel('recall')
    axes[1].invert_yaxis()
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'defect_breakdown.png', dpi=200, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    variant_names = sweep_results_df['name'].astype(str).tolist() if RENDER_ALL_CACHED_VARIANTS else []
    variant_names.extend([str(name) for name in VARIANTS_TO_RENDER])
    variant_names.append(selected_variant_name)
    ordered_variant_names = []
    seen = set()
    for name in variant_names:
        if name not in seen:
            ordered_variant_names.append(name)
            seen.add(name)
    rendered_rows = []
    for variant_name in ordered_variant_names:
        payload = load_variant_outputs(variant_name)
        render_info = render_variant_artifacts(variant_name, payload)
        rendered_rows.append({'variant_name': variant_name, 'plots_dir': render_info['plots_dir'], 'evaluation_dir': render_info['evaluation_dir']})
    rendered_variants_df = pd.DataFrame(rendered_rows)
    display(rendered_variants_df)
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "analysis_df, error_summary_df, defect_recall_df = compute_failure_tables(test_metadata, test_scores_df, threshold)\nanalysis_df.to_csv(results_dir / 'selected_analysis_with_predictions.csv', index=false)\nerror_summary_df.reset_index().to_csv(results_dir / 'selected_error_summary.csv', index=false)\ndefect_recall_df.reset_index().to_csv(results_dir / 'selected_defect_recall.csv', index=false)\ndisplay(error_summary_df)\ndisplay(defect_recall_df)\nfig, axes = plt.subplots(1, 2, figsize=(15, 5))\naxes[0].bar(error_summary_df.index.astype(str), error_summary_df['count'], color=variant_color_anomaly)\naxes[0].set_title(f'prediction outcome counts\\n{selected_variant_name}')\naxes[0].set_ylabel('count')\ntop_defects_df = defect_recall_df.head(10).reset_index()\naxes[1].barh(top_defects_df['defect_type'], top_defects_df['recall'], color=variant_color_defect)\naxes[1].set_xlim(0.0, 1.0)\naxes[1].set_title('top defect-type recall')\naxes[1].set_xlabel('recall')\naxes[1].invert_yaxis()\nplt.tight_layout()\nfig.savefig(plots_dir / 'defect_breakdown.png', dpi=200, bbox_inches='tight')\nplt.show()\nplt.close(fig)\nvariant_names = sweep_results_df['name'].astype(str).tolist() if render_all_cached_variants else []\nvariant_names.extend([str(name) for name in variants_to_render])\nvariant_names.append(selected_variant_name)\nordered_variant_names = []\nseen = set()\nfor name in variant_names:\n    if name not in seen:\n        ordered_variant_names.append(name)\n        seen.add(name)\nrendered_rows = []\nfor variant_name in ordered_variant_names:\n    payload = load_variant_outputs(variant_name)\n    render_info = render_variant_artifacts(variant_name, payload)\n    rendered_rows.append({'variant_name': variant_name, 'plots_dir': render_info['plots_dir'], 'evaluation_dir': render_info['evaluation_dir']})\nrendered_variants_df = pd.dataframe(rendered_rows)\ndisplay(rendered_variants_df)"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Saved Outputs

This cell prints the final artifact locations for the curated review notebook.

In [ ]:
try:
    saved_outputs = {'artifact_root': str(ARTIFACT_ROOT), 'results_dir': str(RESULTS_DIR), 'plots_dir': str(PLOTS_DIR), 'selected_variant_name': selected_variant_name, 'rendered_variants': rendered_variants_df['variant_name'].tolist()}
    saved_outputs
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "saved_outputs = {'artifact_root': str(artifact_root), 'results_dir': str(results_dir), 'plots_dir': str(plots_dir), 'selected_variant_name': selected_variant_name, 'rendered_variants': rendered_variants_df['variant_name'].tolist()}\nsaved_outputs"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise
